# PopDebias-ColdBridge — MovieLens-100K  
**Full benchmark notebook** — trains ALL core baselines + novelty models, self-contained PyTorch, T4 x2 / P100 compatible.

Models trained here: `MostPopular · SASRec · GRU4Rec · BERT4Rec · LLMSeqSim · LLM2SASRec(=BGE2SASRec) · ColdBridge · ColdBridge+Blend · FULL_SYSTEM`

In [ ]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
pip("sentence-transformers>=2.7.0")
print("deps ok")


In [ ]:
import math, random, time, urllib.request, zipfile, os, itertools
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception: pass

if torch.cuda.is_available():
    DEVICE = torch.device("cuda"); N_GPU = torch.cuda.device_count()
    for i in range(N_GPU): print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    DEVICE, N_GPU = torch.device("cpu"), 0; print("CPU only")

WORK_DIR    = Path("/kaggle/working/llmseqrec_ml100k_single_notebook")
DATA_DIR    = WORK_DIR/"data"
EMB_DIR     = WORK_DIR/"embeddings"/"movielens_100k"
RESULTS_DIR = WORK_DIR/"results"/"movielens_100k"
FIG_DIR     = WORK_DIR/"results"/"figures"
for p in [DATA_DIR,EMB_DIR,RESULTS_DIR,FIG_DIR]: p.mkdir(parents=True,exist_ok=True)

DATASET_URL     = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
SESSION_GAP_MIN = 30
MIN_SESSION_LEN = 2
VAL_FRAC, TEST_FRAC = 0.10, 0.20
SEQ_LEN         = 20
EMB_DIM         = 128        # sweet spot for ML-100K (256d overfits)
NUM_HEADS       = 2
NUM_LAYERS      = 2
DROPOUT         = 0.2
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
BATCH_SIZE      = 512 if N_GPU > 0 else 256
MAX_EPOCHS      = 200
PATIENCE        = 15
TOP_K           = 20
CANDIDATE_K     = 500
LONG_TAIL_THR   = 500
TAUS            = [2,3,4,5,6,7,8,10,12,15,20]
DECAYS          = [0.7, 0.8, 0.9]
COLD_ALPHAS     = [0.0, 0.05, 0.1]
BLEND_LAMBDAS   = [0.0, 0.05, 0.10, 0.15, 0.20]  # warm-branch BGE blend weights

# Official published results — used ONLY for comparison printing
PUBLISHED = {
    "SASRec":      {"NDCG@10":0.07303,"HR@10":0.14254,"Coverage":0.36152},
    "LLM2SASRec":  {"NDCG@10":0.05994,"HR@10":0.12472,"Coverage":0.23228},
    "BERT4Rec":    {"NDCG@10":0.05594,"HR@10":0.10913,"Coverage":0.24300},
    "LLM2GRU4Rec": {"NDCG@10":0.05211,"HR@10":0.11136,"Coverage":0.21322},
    "GRU4Rec":     {"NDCG@10":0.04533,"HR@10":0.09577,"Coverage":0.17391},
    "LLMSeqSim":   {"NDCG@10":0.01228,"HR@10":0.02895,"Coverage":0.65694},
    "MostPopular": {"NDCG@10":0.01204,"HR@10":0.03118,"Coverage":0.02501},
}
print("Config ready — DEVICE:", DEVICE, "| EMB_DIM:", EMB_DIM)


In [ ]:
def download_ml100k():
    udata = DATA_DIR/"ml-100k"/"u.data"; uitem = DATA_DIR/"ml-100k"/"u.item"
    if udata.exists() and uitem.exists(): return udata, uitem
    arch = DATA_DIR/"ml-100k.zip"
    if not arch.exists():
        print("Downloading MovieLens-100K …")
        urllib.request.urlretrieve(DATASET_URL, arch)
    with zipfile.ZipFile(arch) as zf: zf.extractall(DATA_DIR)
    return udata, uitem

udata_path, uitem_path = download_ml100k()
raw = pd.read_csv(udata_path, sep="\t", names=["UserId","ItemId","Rating","Timestamp"])
raw["Timestamp"] = raw["Timestamp"].astype(int)
raw["ItemId"]    = raw["ItemId"].astype(int)
raw = raw.sort_values(["UserId","Timestamp","ItemId"]).reset_index(drop=True)

gap_sec = SESSION_GAP_MIN * 60
user_ch = raw["UserId"].ne(raw["UserId"].shift(1))
ts_gap  = raw["Timestamp"].diff().fillna(0)
raw["SessionId"] = (user_ch|(ts_gap>gap_sec)).cumsum().astype(int)
sz = raw.groupby("SessionId").size()
raw = raw[raw["SessionId"].isin(sz[sz>=MIN_SESSION_LEN].index)].copy()
raw["SessionId"] = raw["SessionId"].map({o:i+1 for i,o in enumerate(sorted(raw["SessionId"].unique()))})
raw["Time"] = pd.to_datetime(raw["Timestamp"], unit="s")

sess_end = raw.groupby("SessionId")["Time"].max().sort_values()
sids = sess_end.index.to_numpy(); n = len(sids)
n_test = max(1,round(n*TEST_FRAC)); n_val = max(1,round(n*VAL_FRAC))
train_ids = set(sids[:n-n_val-n_test])
val_ids   = set(sids[n-n_val-n_test:n-n_test])
test_ids  = set(sids[n-n_test:])
train_df = raw[raw["SessionId"].isin(train_ids)].copy()
val_df   = raw[raw["SessionId"].isin(val_ids)].copy()
test_df  = raw[raw["SessionId"].isin(test_ids)].copy()

def build_cases(df):
    prompts,gts,lengths = {},{},{}
    for sid,g in df.groupby("SessionId"):
        items = g.sort_values("Time")["ItemId"].to_numpy(dtype=int)
        if len(items)<2: continue
        sid=int(sid); prompts[sid]=items[:-1]; gts[sid]=items[-1:]; lengths[sid]=len(items)-1
    return prompts,gts,lengths

train_prompts,train_gts,train_lengths = build_cases(train_df)
val_prompts,  val_gts,  val_lengths   = build_cases(val_df)
test_prompts, test_gts, test_lengths  = build_cases(test_df)

all_item_ids = np.array(sorted(raw["ItemId"].unique()),dtype=int)
num_items    = len(all_item_ids)
item2idx  = {int(v):i   for i,v in enumerate(all_item_ids)}
item2red  = {int(v):i+1 for i,v in enumerate(all_item_ids)}
red2item  = {i+1:int(v) for i,v in enumerate(all_item_ids)}
item_counts   = train_df["ItemId"].value_counts().to_dict()
popular_top10 = [int(i) for i,_ in sorted(item_counts.items(),key=lambda kv:kv[1],reverse=True)][:10]

print(f"Sessions: train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}")
print(f"Items:{num_items}  test_prompts:{len(test_prompts)}")
print(f"Train interactions:{len(train_df)}  Test:{len(test_df)}")
print(f"Avg test seq len: {np.mean([len(v) for v in test_prompts.values()]):.1f}")


In [ ]:
GENRE_COLS=["unknown","action","adventure","animation","childrens","comedy",
            "crime","documentary","drama","fantasy","film_noir","horror",
            "musical","mystery","romance","sci_fi","thriller","war","western"]
cols=["ItemId","title","release_date","video_release_date","imdb_url",*GENRE_COLS]
meta_df=pd.read_csv(uitem_path,sep="|",names=cols,encoding="latin-1",
                    usecols=list(range(len(cols))),engine="python")
meta_df["text"]=meta_df.apply(
    lambda r: f"{r['title']}. Genres: {', '.join(g.replace('_',' ') for g in GENRE_COLS if int(r[g])==1) or 'unknown'}.",
    axis=1)
text_lookup=meta_df.set_index("ItemId")["text"].to_dict()

EMB_PATH=EMB_DIR/"item_embeddings_bge_m3.npy"
if EMB_PATH.exists():
    cached=np.load(EMB_PATH)
    if cached.shape[0]!=num_items: EMB_PATH.unlink(); cached=None
else: cached=None

if cached is None:
    print("Encoding items with BGE-M3 …")
    texts=[text_lookup.get(int(i),f"Movie {int(i)}.") for i in all_item_ids]
    m_bge=SentenceTransformer("BAAI/bge-m3",device="cuda" if N_GPU>0 else "cpu")
    raw_emb=m_bge.encode(texts,batch_size=256,normalize_embeddings=True,
                          show_progress_bar=True,convert_to_numpy=True).astype(np.float32)
    np.save(EMB_PATH,raw_emb); del m_bge; torch.cuda.empty_cache()
    print(f"Saved: {raw_emb.shape}")
else:
    raw_emb=cached; print(f"Cached: {raw_emb.shape}")

norms=np.linalg.norm(raw_emb,axis=1,keepdims=True); norms[norms==0]=1.0
bge_emb=(raw_emb/norms).astype(np.float32)
count_arr=np.array([float(item_counts.get(int(i),1)) for i in all_item_ids],dtype=np.float32)
print(f"BGE-M3: {bge_emb.shape}")

# PCA reduce once — used by all LLM-init models
print(f"PCA 1024d → {EMB_DIM}d …")
pca=PCA(n_components=EMB_DIM,random_state=SEED)
bge_r=pca.fit_transform(bge_emb).astype(np.float32)
nr=np.linalg.norm(bge_r,axis=1,keepdims=True); nr[nr==0]=1.0; bge_r/=nr
print(f"  Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")


In [ ]:
# ── SASRec (self-attention) ───────────────────────────────────────────────────
class SASRec(nn.Module):
    """2-layer causal self-attention. Pre-norm, GELU, tied output."""
    def __init__(self,num_items,seq_len,emb_dim,num_heads=2,num_layers=2,dropout=0.2):
        super().__init__()
        self.item_emb=nn.Embedding(num_items+1,emb_dim,padding_idx=0)
        self.pos_emb =nn.Embedding(seq_len,emb_dim)
        self.drop_in =nn.Dropout(dropout)
        enc=nn.TransformerEncoderLayer(d_model=emb_dim,nhead=num_heads,
            dim_feedforward=emb_dim*4,dropout=dropout,activation="gelu",
            batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_layers)
        self.norm_out=nn.LayerNorm(emb_dim)
        nn.init.normal_(self.item_emb.weight,std=0.02)
        nn.init.normal_(self.pos_emb.weight,std=0.02)
        for m in self.transformer.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self,x):
        B,L=x.shape
        h=self.drop_in(self.item_emb(x)+self.pos_emb(torch.arange(L,device=x.device).unsqueeze(0)))
        causal=torch.triu(torch.ones(L,L,dtype=torch.bool,device=x.device),diagonal=1)
        h=self.transformer(h,mask=causal,src_key_padding_mask=(x==0),is_causal=True)
        h=self.norm_out(h[:,-1,:])
        return h@self.item_emb.weight[1:].T
    def load_pretrained(self,emb):
        with torch.no_grad(): self.item_emb.weight[1:]=torch.tensor(emb,dtype=torch.float32)

# ── GRU4Rec ───────────────────────────────────────────────────────────────────
class GRU4Rec(nn.Module):
    """2-layer GRU sequential model."""
    def __init__(self,num_items,emb_dim,hidden_dim=128,num_layers=2,dropout=0.2):
        super().__init__()
        self.item_emb=nn.Embedding(num_items+1,emb_dim,padding_idx=0)
        self.gru=nn.GRU(emb_dim,hidden_dim,num_layers=num_layers,
                         batch_first=True,dropout=dropout if num_layers>1 else 0.0)
        self.out_proj=nn.Linear(hidden_dim,num_items,bias=False)
        self.drop=nn.Dropout(dropout)
        nn.init.normal_(self.item_emb.weight,std=0.02)
        nn.init.xavier_uniform_(self.out_proj.weight)
    def forward(self,x):
        # x: (B, seq_len) — use last non-padding hidden state
        emb=self.drop(self.item_emb(x))           # (B,L,D)
        lengths=((x!=0).sum(dim=1).clamp(min=1))  # (B,)
        packed=nn.utils.rnn.pack_padded_sequence(
            emb,lengths.cpu(),batch_first=True,enforce_sorted=False)
        _,h=self.gru(packed)                       # h: (num_layers,B,H)
        h=h[-1]                                    # last layer: (B,H)
        return self.out_proj(self.drop(h))         # (B, num_items)

# ── BERT4Rec ─────────────────────────────────────────────────────────────────
class BERT4Rec(nn.Module):
    """Bidirectional transformer with masked item prediction.
       At inference, mask the last item and predict it."""
    def __init__(self,num_items,seq_len,emb_dim,num_heads=2,num_layers=2,dropout=0.2):
        super().__init__()
        self.seq_len=seq_len
        self.mask_token=num_items+1          # special [MASK] token id
        self.item_emb=nn.Embedding(num_items+2,emb_dim,padding_idx=0)  # +2 for mask
        self.pos_emb =nn.Embedding(seq_len,emb_dim)
        self.drop_in =nn.Dropout(dropout)
        enc=nn.TransformerEncoderLayer(d_model=emb_dim,nhead=num_heads,
            dim_feedforward=emb_dim*4,dropout=dropout,activation="gelu",
            batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=num_layers)
        self.norm_out=nn.LayerNorm(emb_dim)
        nn.init.normal_(self.item_emb.weight,std=0.02)
        nn.init.normal_(self.pos_emb.weight,std=0.02)
    def forward(self,x):
        B,L=x.shape
        h=self.drop_in(self.item_emb(x)+self.pos_emb(torch.arange(L,device=x.device).unsqueeze(0)))
        h=self.transformer(h,src_key_padding_mask=(x==0))
        h=self.norm_out(h)
        # Use only real item embeddings for output projection (exclude mask token)
        W=self.item_emb.weight[1:self.mask_token]  # (num_items, emb_dim)
        return h@W.T                                # (B,L,num_items)
    def predict_last(self,x):
        """Inference: replace last non-padding token with MASK, return logits for that position."""
        x=x.clone()
        lengths=(x!=0).sum(dim=1).clamp(min=1)
        for i,l in enumerate(lengths): x[i,l-1]=self.mask_token
        logits=self.forward(x)  # (B,L,num_items)
        # gather the masked position
        idx=(lengths-1).unsqueeze(-1).unsqueeze(-1).expand(-1,1,logits.size(-1))
        return logits.gather(1,idx).squeeze(1)  # (B, num_items)

print("Models defined: SASRec, GRU4Rec, BERT4Rec")


In [ ]:
def build_train_examples(df,i2r,seq_len):
    xs,ys=[],[]
    for _,g in df.groupby("SessionId"):
        items=[i2r[int(i)] for i in g.sort_values("Time")["ItemId"] if int(i) in i2r]
        if len(items)<2: continue
        for t in range(1,len(items)):
            prompt=items[max(0,t-seq_len):t]
            x=np.zeros(seq_len,dtype=np.int32); x[-len(prompt):]=prompt
            xs.append(x); ys.append(items[t]-1)
    return np.array(xs,np.int32),np.array(ys,np.int64)

def build_bert4rec_examples(df,i2r,seq_len,mask_prob=0.2):
    """BERT4Rec masked training: randomly mask items, predict them."""
    xs,ys=[],[]
    rng=np.random.default_rng(SEED)
    for _,g in df.groupby("SessionId"):
        items=[i2r[int(i)] for i in g.sort_values("Time")["ItemId"] if int(i) in i2r]
        if len(items)<2: continue
        seq=items[-seq_len:]
        x=np.zeros(seq_len,dtype=np.int32); x[-len(seq):]=seq
        # create multiple masked versions
        for pos in range(seq_len-len(seq),seq_len):
            if x[pos]==0: continue
            if rng.random()<mask_prob:
                xi=x.copy(); yi=np.full(seq_len,-1,dtype=np.int64)
                yi[pos]=xi[pos]-1   # 0-based label
                xi[pos]=len(i2r)+1  # mask token id (num_items+1)
                xs.append(xi); ys.append(yi)
    if not xs: return build_train_examples(df,i2r,seq_len)  # fallback
    return np.array(xs,np.int32),np.array(ys,np.int64)

class _DS(torch.utils.data.Dataset):
    def __init__(self,x,y): self.x=torch.tensor(x,torch.long); self.y=torch.tensor(y,torch.long)
    def __len__(self): return len(self.x)
    def __getitem__(self,i): return self.x[i],self.y[i]

def train_model(model,x_tr,y_tr,max_epochs=MAX_EPOCHS,batch_size=BATCH_SIZE,
                lr=LR,wd=WEIGHT_DECAY,patience=PATIENCE,val_frac=0.1,
                device=DEVICE,is_bert=False):
    n_val=max(1,int(len(x_tr)*val_frac)); idx=np.random.permutation(len(x_tr))
    vi,ti=idx[:n_val],idx[n_val:]
    kw=dict(num_workers=2,pin_memory=(device.type=="cuda"))
    dl_tr =torch.utils.data.DataLoader(_DS(x_tr[ti],y_tr[ti]),batch_size=batch_size,shuffle=True,**kw)
    dl_val=torch.utils.data.DataLoader(_DS(x_tr[vi],y_tr[vi]),batch_size=batch_size*2,shuffle=False,**kw)
    net=nn.DataParallel(model) if device.type=="cuda" and torch.cuda.device_count()>1 else model
    net=net.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd,betas=(0.9,0.98))
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max_epochs,eta_min=lr*0.01)
    crit=nn.CrossEntropyLoss(ignore_index=-1)
    best_val,no_imp,best_state=float("inf"),0,None
    t0=time.perf_counter()
    for ep in range(1,max_epochs+1):
        net.train()
        for xb,yb in dl_tr:
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad(set_to_none=True)
            if is_bert:
                logits=net(xb)  # (B,L,num_items)
                loss=crit(logits.reshape(-1,logits.size(-1)),yb.reshape(-1))
            else:
                loss=crit(net(xb),yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        sched.step()
        net.eval(); vl=0.0
        with torch.no_grad():
            for xb,yb in dl_val:
                xb,yb=xb.to(device),yb.to(device)
                if is_bert:
                    logits=net(xb); vl+=crit(logits.reshape(-1,logits.size(-1)),yb.reshape(-1)).item()*len(xb)
                else:
                    vl+=crit(net(xb),yb).item()*len(xb)
        vl/=len(dl_val.dataset)
        if vl<best_val-1e-4: best_val=vl; no_imp=0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            no_imp+=1
            if no_imp>=patience: print(f"  Early stop ep={ep} best={best_val:.4f}"); break
        if ep%25==0: print(f"  ep {ep:3d}  val={vl:.4f}  best={best_val:.4f}")
    if best_state: model.load_state_dict(best_state)
    return time.perf_counter()-t0, ep

@torch.no_grad()
def predict(model,prompts,i2r,r2i,seq_len,batch_size=2048,device=DEVICE,is_bert=False):
    sids,xs,seen_sets=[],[],[]
    for sid,prompt in prompts.items():
        red=[i2r[int(i)] for i in prompt if int(i) in i2r]
        if not red: continue
        x=np.zeros(seq_len,np.int32); x[-len(red[-seq_len:]):]=red[-seq_len:]
        sids.append(int(sid)); xs.append(x); seen_sets.append(set(red))
    if not sids: return {}
    model.eval()
    xt=torch.tensor(np.array(xs),dtype=torch.long)
    all_logits=[]
    for i in range(0,len(xt),batch_size):
        batch=xt[i:i+batch_size].to(device)
        if is_bert: all_logits.append(model.predict_last(batch).cpu().numpy())
        else:       all_logits.append(model(batch).cpu().numpy())
    all_logits=np.concatenate(all_logits,axis=0)
    out={}
    for i,sid in enumerate(sids):
        sc=all_logits[i].copy()
        for r in seen_sets[i]:
            j=r-1
            if 0<=j<len(sc): sc[j]=-1e9
        order=np.argsort(sc)[::-1]
        orig=np.array([r2i[j+1] for j in order if (j+1) in r2i],dtype=np.int32)
        out[sid]=(orig,sc[order[:len(orig)]])
    return out

print("Training & inference defined.")


In [ ]:
def ndcg_hr(preds,gts,k):
    nd,hr=[],[]
    for sid,gt in gts.items():
        tgt=int(gt[0]); rec=[int(i) for i in preds.get(sid,[])[:k]]
        if tgt in rec: hr.append(1.0); nd.append(1.0/math.log2(rec.index(tgt)+2))
        else: hr.append(0.0); nd.append(0.0)
    return float(np.mean(nd)),float(np.mean(hr))

def lt_hr(preds,gts,k=10):
    hits=tot=0
    for sid,gt in gts.items():
        tgt=int(gt[0])
        if item_counts.get(tgt,0)>=LONG_TAIL_THR: continue
        tot+=1
        if tgt in [int(i) for i in preds.get(sid,[])[:k]]: hits+=1
    return hits/tot if tot else 0.0

def cov(preds,k=10):
    used=set()
    for rec in preds.values(): used.update(int(i) for i in rec[:k])
    return len(used)/num_items if num_items else 0.0

def seren(preds,gts,k=10):
    exp=set(popular_top10); vals=[]
    for sid,gt in gts.items():
        tgt=int(gt[0]); rec=[int(i) for i in preds.get(sid,[])[:k]]
        vals.append(1.0 if (tgt in rec and tgt not in exp) else 0.0)
    return float(np.mean(vals))

def ild_metric(preds,k=10):
    vals=[]
    for rec in preds.values():
        ids=[int(i) for i in rec[:k] if int(i) in item2idx]
        if len(ids)<2: continue
        vecs=bge_emb[np.array([item2idx[i] for i in ids])]
        sims=vecs@vecs.T; ui=np.triu_indices(len(ids),k=1)
        if len(ui[0]): vals.append(float(np.mean(1.0-sims[ui])))
    return float(np.mean(vals)) if vals else 0.0

def log_row(name,preds,gts,alpha=None,tau=None,train_t=0.0,infer_t=0.0,extra=None):
    n10,h10=ndcg_hr(preds,gts,10); n20,h20=ndcg_hr(preds,gts,20)
    row={"model_name":name,"alpha":alpha,"tau":tau,
         "NDCG@10":n10,"NDCG@20":n20,"HR@10":h10,"HR@20":h20,
         "LongTail_HR@10":lt_hr(preds,gts),"CatalogCoverage":cov(preds),
         "Serendipity":seren(preds,gts),"ILD@10":ild_metric(preds),
         "training_time_sec":train_t,"inference_time_sec":infer_t}
    if extra: row.update(extra)
    pub=PUBLISHED.get("SASRec",{}); ref_nd=pub.get("NDCG@10",0.073); ref_hr=pub.get("HR@10",0.143)
    d_nd=(n10-ref_nd)/ref_nd*100; d_hr=(h10-ref_hr)/ref_hr*100
    b_nd="✓" if n10>ref_nd else "✗"; b_hr="✓" if h10>ref_hr else "✗"
    print(f"  [{name}]  NDCG@10={n10:.5f}({b_nd}{d_nd:+.1f}%)  HR@10={h10:.5f}({b_hr}{d_hr:+.1f}%)  Cov={row['CatalogCoverage']:.3f}")
    return row

print("Metrics defined.")


In [ ]:
def sess_emb_fn(prompt,decay=1.0):
    idxs=[item2idx[int(i)] for i in prompt if int(i) in item2idx]
    if not idxs: return None
    vecs=bge_emb[np.array(idxs)]
    if decay<1.0 and len(vecs)>1:
        n=len(vecs); w=np.array([decay**(n-k-1) for k in range(n)],np.float32); w/=w.sum()
        v=np.average(vecs,axis=0,weights=w)
    else: v=vecs.mean(axis=0)
    norm=np.linalg.norm(v); return v/norm if norm>0 else v

def cold_pred(prompt,top_k,decay=0.8,alpha=0.05):
    q=sess_emb_fn(prompt,decay)
    if q is None: return all_item_ids[:top_k].astype(int)
    sc=bge_emb@q
    if alpha>0: sc=sc*(1.0/np.power(np.maximum(count_arr,1.0),alpha))
    seen=[item2idx[int(i)] for i in prompt if int(i) in item2idx]
    if seen: sc[np.array(seen)]=-1e9
    return all_item_ids[np.argsort(sc)[::-1][:top_k]].astype(int)

def slot_fill(ranked,top_k=TOP_K,protected=5,div_slots=5,tail_thr=200):
    ranked=[int(i) for i in ranked]; final=list(dict.fromkeys(ranked[:protected]))
    remain=[i for i in ranked if i not in final]
    tail=[i for i in remain if item_counts.get(i,0)<tail_thr]
    head=[i for i in remain if item_counts.get(i,0)>=tail_thr]
    slots=top_k-len(final); n_t=min(div_slots,len(tail),slots)
    out,ti,hi,pos=[],0,0,0
    while len(out)<slots:
        if pos%3==0 and ti<n_t: out.append(tail[ti]); ti+=1
        elif hi<len(head): out.append(head[hi]); hi+=1
        elif ti<len(tail): out.append(tail[ti]); ti+=1
        else: break
        pos+=1
    for i in remain:
        if len(out)>=slots: break
        if i not in out: out.append(i)
    final.extend(out); return final[:top_k]

def llmseqsim_recs(prompt,top_n=100):
    q=sess_emb_fn(prompt)
    if q is None: return list(all_item_ids[:top_n].astype(int))
    sc=bge_emb@q; seen=[item2idx[int(i)] for i in prompt if int(i) in item2idx]
    if seen: sc[np.array(seen)]=-1e9
    return [int(all_item_ids[j]) for j in np.argsort(sc)[::-1][:top_n]]

def z_score(arr):
    s=arr.std(); return (arr-arr.mean())/s if s>1e-9 else np.zeros_like(arr)

def full_system(prompts,warm_items,warm_sc,tau,decay,cold_alpha,
                blend_lambda=0.0,top_k=TOP_K,
                protected=None,div_slots=None,tail_thr=200,seq_positions=None):
    """
    blend_lambda: weight for BGE cosine blend on warm branch.
      0.0 = pure model logits (original ColdBridge)
      0.1 = 90% model + 10% BGE sim (ColdBridge+Blend — higher HR)
    """
    out={}
    for sid,prompt in prompts.items():
        if len(prompt)<tau:
            out[sid]=cold_pred(prompt,top_k,decay,cold_alpha); continue
        items=warm_items.get(sid,np.array([],np.int32))
        sc=warm_sc.get(sid,np.array([],np.float32))
        if len(items)==0: out[sid]=cold_pred(prompt,top_k,decay,cold_alpha); continue

        # Optional warm-branch score blend with BGE similarity
        if blend_lambda>0.0:
            q=sess_emb_fn(prompt)
            if q is not None:
                # compute BGE sim for only these candidates
                idxs=np.array([item2idx[int(i)] for i in items if int(i) in item2idx],dtype=int)
                if len(idxs)==len(items):
                    bge_sim=bge_emb[idxs]@q
                    sc=((1.0-blend_lambda)*z_score(sc.astype(np.float64))
                        +blend_lambda*z_score(bge_sim.astype(np.float64)))

        order=np.argsort(sc)[::-1]; ranked=[int(items[j]) for j in order]
        if protected is not None and div_slots is not None:
            ranked=slot_fill(ranked,top_k,protected,div_slots,tail_thr)
        else: ranked=ranked[:top_k]
        if seq_positions is not None:
            llm=llmseqsim_recs(prompt,top_n=top_k*2)
            final=list(dict.fromkeys(ranked[:seq_positions]))
            for i in llm:
                if len(final)>=top_k: break
                if i not in final: final.append(i)
            ranked=final[:top_k]
        out[sid]=np.array(ranked[:top_k],np.int32)
    return out

print("Novelty functions defined.")


In [ ]:
rows=[]; abl=[]

# ── MostPopular ───────────────────────────────────────────────────────────────
print("[BASELINE] MostPopular")
pop_s=[int(i) for i,_ in sorted(item_counts.items(),key=lambda kv:kv[1],reverse=True)]
def pop_preds(p,k): return {s:np.array([i for i in pop_s if i not in {int(x) for x in pr}][:k],np.int32) for s,pr in p.items()}
t0=time.perf_counter()
rp=log_row("MostPopular",pop_preds(test_prompts,TOP_K),test_gts,infer_t=time.perf_counter()-t0)
rows.append(rp); abl.append({"step":0,"model":"MostPopular","NDCG@10":rp["NDCG@10"],"HR@10":rp["HR@10"]})

# ── LLMSeqSim (zero training — pure BGE content similarity) ──────────────────
print("\n[BASELINE] LLMSeqSim (BGE content similarity, no training)")
t0=time.perf_counter()
llmss_preds={}
for sid,prompt in test_prompts.items():
    llmss_preds[sid]=np.array(llmseqsim_recs(prompt,top_n=TOP_K),dtype=np.int32)
rl=log_row("LLMSeqSim",llmss_preds,test_gts,train_t=0.0,infer_t=time.perf_counter()-t0)
rows.append(rl); abl.append({"step":1,"model":"LLMSeqSim","NDCG@10":rl["NDCG@10"],"HR@10":rl["HR@10"]})

# ── Build training examples once ─────────────────────────────────────────────
print("\nBuilding training examples …")
x_tr,y_tr=build_train_examples(train_df,item2red,SEQ_LEN)
print(f"  SASRec/GRU examples: {len(x_tr)}")
x_tr_b,y_tr_b=build_bert4rec_examples(train_df,item2red,SEQ_LEN,mask_prob=0.2)
print(f"  BERT4Rec examples:   {len(x_tr_b)}")

# ── SASRec (vanilla — no pretrained embeddings) ───────────────────────────────
print("\n[BASELINE] SASRec (vanilla)")
sas_vanilla=SASRec(num_items,SEQ_LEN,EMB_DIM,NUM_HEADS,NUM_LAYERS,DROPOUT)
sas_t,sas_ep=train_model(sas_vanilla,x_tr,y_tr,device=DEVICE)
print(f"  Done: {sas_ep} epochs in {sas_t:.0f}s")
t0=time.perf_counter()
sas_test_out=predict(sas_vanilla,test_prompts,item2red,red2item,SEQ_LEN,device=DEVICE)
sas_preds={s:it[:TOP_K] for s,(it,_) in sas_test_out.items()}
rsas=log_row("SASRec",sas_preds,test_gts,train_t=sas_t,infer_t=time.perf_counter()-t0)
rows.append(rsas); abl.append({"step":2,"model":"SASRec","NDCG@10":rsas["NDCG@10"],"HR@10":rsas["HR@10"]})

# ── GRU4Rec ───────────────────────────────────────────────────────────────────
print("\n[BASELINE] GRU4Rec")
gru=GRU4Rec(num_items,EMB_DIM,hidden_dim=EMB_DIM,num_layers=2,dropout=DROPOUT)
gru_t,gru_ep=train_model(gru,x_tr,y_tr,device=DEVICE)
print(f"  Done: {gru_ep} epochs in {gru_t:.0f}s")
t0=time.perf_counter()
gru_test_out=predict(gru,test_prompts,item2red,red2item,SEQ_LEN,device=DEVICE)
gru_preds={s:it[:TOP_K] for s,(it,_) in gru_test_out.items()}
rgru=log_row("GRU4Rec",gru_preds,test_gts,train_t=gru_t,infer_t=time.perf_counter()-t0)
rows.append(rgru); abl.append({"step":3,"model":"GRU4Rec","NDCG@10":rgru["NDCG@10"],"HR@10":rgru["HR@10"]})

# ── BERT4Rec ──────────────────────────────────────────────────────────────────
print("\n[BASELINE] BERT4Rec")
bert=BERT4Rec(num_items,SEQ_LEN,EMB_DIM,NUM_HEADS,NUM_LAYERS,DROPOUT)
bert_t,bert_ep=train_model(bert,x_tr_b,y_tr_b,device=DEVICE,is_bert=True)
print(f"  Done: {bert_ep} epochs in {bert_t:.0f}s")
t0=time.perf_counter()
bert_test_out=predict(bert,test_prompts,item2red,red2item,SEQ_LEN,device=DEVICE,is_bert=True)
bert_preds={s:it[:TOP_K] for s,(it,_) in bert_test_out.items()}
rbert=log_row("BERT4Rec",bert_preds,test_gts,train_t=bert_t,infer_t=time.perf_counter()-t0)
rows.append(rbert); abl.append({"step":4,"model":"BERT4Rec","NDCG@10":rbert["NDCG@10"],"HR@10":rbert["HR@10"]})

print("\nAll baselines done.")


In [ ]:
# ── BGE2SASRec (SASRec + BGE-M3 init — LLM2SASRec equivalent) ────────────────
print("\n[NOVELTY BACKBONE] BGE2SASRec (LLM2SASRec equivalent, free)")
backbone=SASRec(num_items,SEQ_LEN,EMB_DIM,NUM_HEADS,NUM_LAYERS,DROPOUT)
backbone.load_pretrained(bge_r)
train_t,last_ep=train_model(backbone,x_tr,y_tr,device=DEVICE)
print(f"  Done: {last_ep} epochs in {train_t:.0f}s")

t0=time.perf_counter()
val_out =predict(backbone,val_prompts, item2red,red2item,SEQ_LEN,device=DEVICE)
test_out=predict(backbone,test_prompts,item2red,red2item,SEQ_LEN,device=DEVICE)
infer_t=time.perf_counter()-t0

val_it ={s:it for s,(it,_) in val_out.items()}
val_sc ={s:sc for s,(_,sc) in val_out.items()}
test_it={s:it for s,(it,_) in test_out.items()}
test_sc={s:sc for s,(_,sc) in test_out.items()}

bge_preds={s:it[:TOP_K] for s,it in test_it.items()}
rb=log_row("BGE2SASRec",bge_preds,test_gts,train_t=train_t,infer_t=infer_t,
            extra={"embedding":"BGE-M3"})
rows.append(rb)
abl.append({"step":5,"model":"BGE2SASRec","NDCG@10":rb["NDCG@10"],"HR@10":rb["HR@10"]})

if rb["NDCG@10"]<0.030:
    print(f"  WARNING: NDCG={rb['NDCG@10']:.5f} < 0.030 — check data split.")
else:
    print(f"  Backbone PASS: NDCG={rb['NDCG@10']:.5f}")


In [ ]:
# ── ColdBridge grid: tau × decay × cold_alpha ────────────────────────────────
print("\n[NOVELTY] ColdBridge grid search …")
best_cb_cfg=None; best_cb_nd=-1.0; cb_log=[]
for tau,decay,calpha in itertools.product(TAUS,DECAYS,COLD_ALPHAS):
    vp=full_system(val_prompts,val_it,val_sc,tau=tau,decay=decay,cold_alpha=calpha)
    nd,hr=ndcg_hr(vp,val_gts,10); cv=cov(vp)
    cb_log.append({"tau":tau,"decay":decay,"alpha":calpha,"NDCG@10":nd,"HR@10":hr,"Coverage":cv})
    if nd>best_cb_nd: best_cb_nd=nd; best_cb_cfg={"tau":tau,"decay":decay,"cold_alpha":calpha}

print(f"  Best ColdBridge: {best_cb_cfg}  val_NDCG={best_cb_nd:.5f}")
t0=time.perf_counter()
cb_pred=full_system(test_prompts,test_it,test_sc,**best_cb_cfg)
rc=log_row("ColdBridge",cb_pred,test_gts,tau=best_cb_cfg["tau"],
            train_t=train_t,infer_t=time.perf_counter()-t0,
            extra={"cold_decay":best_cb_cfg["decay"],"cold_alpha":best_cb_cfg["cold_alpha"],"blend_lambda":0.0})
rows.append(rc); abl.append({"step":6,"model":"ColdBridge","NDCG@10":rc["NDCG@10"],"HR@10":rc["HR@10"]})

# ── ColdBridge + BGE Blend: tau × decay × cold_alpha × blend_lambda ──────────
print("\n[NOVELTY] ColdBridge + BGE Blend grid search (targets HR@10 improvement) …")
best_bl_cfg=None; best_bl_joint=-1.0
for tau,decay,calpha,lam in itertools.product(TAUS,DECAYS,COLD_ALPHAS,BLEND_LAMBDAS):
    if lam==0.0: continue  # already covered by ColdBridge above
    vp=full_system(val_prompts,val_it,val_sc,tau=tau,decay=decay,
                    cold_alpha=calpha,blend_lambda=lam)
    nd,hr=ndcg_hr(vp,val_gts,10)
    # joint: balance NDCG and HR equally — we want to improve HR without losing NDCG
    joint=0.5*nd+0.5*hr
    if joint>best_bl_joint: best_bl_joint=joint; best_bl_cfg={"tau":tau,"decay":decay,"cold_alpha":calpha,"blend_lambda":lam}

print(f"  Best Blend: {best_bl_cfg}  val_joint={best_bl_joint:.5f}")
t0=time.perf_counter()
bl_pred=full_system(test_prompts,test_it,test_sc,**best_bl_cfg)
rbl=log_row("ColdBridge+Blend",bl_pred,test_gts,tau=best_bl_cfg["tau"],
             train_t=train_t,infer_t=time.perf_counter()-t0,
             extra={k:v for k,v in best_bl_cfg.items()})
rows.append(rbl); abl.append({"step":7,"model":"ColdBridge+Blend","NDCG@10":rbl["NDCG@10"],"HR@10":rbl["HR@10"]})

# ── SlotFill on best blend config ─────────────────────────────────────────────
print("\n[NOVELTY] Slot-fill grid …")
base_nd_bl=ndcg_hr(full_system(val_prompts,val_it,val_sc,**best_bl_cfg),val_gts,10)[0]
best_sf_cfg=None; best_sf_nd=-1.0
for prot,div,tthr in itertools.product([3,5],[3,5,7],[100,200,500]):
    vp=full_system(val_prompts,val_it,val_sc,**best_bl_cfg,protected=prot,div_slots=div,tail_thr=tthr)
    nd=ndcg_hr(vp,val_gts,10)[0]
    if nd+1e-9<base_nd_bl: continue
    if nd>best_sf_nd: best_sf_nd=nd; best_sf_cfg={"protected":prot,"div_slots":div,"tail_thr":tthr}
if best_sf_cfg is None: best_sf_cfg={"protected":5,"div_slots":5,"tail_thr":200}
print(f"  Best SlotFill: {best_sf_cfg}")

# ── Hybrid ensemble on best blend+slotfill config ─────────────────────────────
print("\n[NOVELTY] Hybrid ensemble grid …")
best_hy_cfg=None; best_hy_joint=-1.0
for sp in [10,12,15,17]:
    vp=full_system(val_prompts,val_it,val_sc,**best_bl_cfg,**best_sf_cfg,seq_positions=sp)
    nd,hr=ndcg_hr(vp,val_gts,10); joint=0.5*nd+0.5*hr
    if joint>best_hy_joint: best_hy_joint=joint; best_hy_cfg={"seq_positions":sp}
print(f"  Best hybrid: {best_hy_cfg}")

# ── FULL_SYSTEM ───────────────────────────────────────────────────────────────
t0=time.perf_counter()
hy_pred=full_system(test_prompts,test_it,test_sc,**best_bl_cfg,**best_sf_cfg,**best_hy_cfg)
rh=log_row("FULL_SYSTEM",hy_pred,test_gts,tau=best_bl_cfg["tau"],
            train_t=train_t,infer_t=time.perf_counter()-t0,
            extra={"debias_variant":"blend+slotfill+hybrid",**best_bl_cfg,**best_sf_cfg,**best_hy_cfg})
rows.append(rh); abl.append({"step":8,"model":"FULL_SYSTEM","NDCG@10":rh["NDCG@10"],"HR@10":rh["HR@10"]})

# tau sweep figure
cb_df=pd.DataFrame(cb_log)
tau_b=cb_df.sort_values("NDCG@10",ascending=False).groupby("tau",as_index=False).first()
plt.figure(figsize=(7,4))
plt.plot(tau_b["tau"],tau_b["NDCG@10"],"o-",label="NDCG@10")
plt.plot(tau_b["tau"],tau_b["HR@10"],"s-",label="HR@10")
plt.xlabel("tau"); plt.title("ColdBridge tau sweep (val)"); plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/"tau_sweep_v3.png",dpi=150); plt.close()


In [ ]:
print("\n" + "="*70)
print("COMPREHENSIVE COMPARISON TABLE")
print("="*70)

# Build comparison rows
pub = PUBLISHED
all_results = {r["model_name"]: r for r in rows}

header = f"{'Model':<25} {'NDCG@10':>9} {'HR@10':>9} {'Coverage':>9} {'vs SASRec NDCG':>15} {'vs SASRec HR':>13}"
print(header); print("-"*70)

sas_nd = pub["SASRec"]["NDCG@10"]; sas_hr = pub["SASRec"]["HR@10"]

# Published core models (reference)
for name in ["MostPopular","LLMSeqSim","GRU4Rec","BERT4Rec","LLM2SASRec","SASRec"]:
    if name in all_results:
        r = all_results[name]
        nd=float(r["NDCG@10"]); hr=float(r["HR@10"]); cv=float(r["CatalogCoverage"])
    elif name in pub:
        nd=pub[name]["NDCG@10"]; hr=pub[name]["HR@10"]; cv=pub[name]["Coverage"]
    else: continue
    d_nd=(nd-sas_nd)/sas_nd*100; d_hr=(hr-sas_hr)/sas_hr*100
    b_nd="✓" if nd>sas_nd else " "; b_hr="✓" if hr>sas_hr else " "
    print(f"  {name:<23} {nd:>9.5f} {hr:>9.5f} {cv:>9.3f}   {b_nd}{d_nd:>+8.1f}%   {b_hr}{d_hr:>+8.1f}%")

print("-"*70)
# Our models
for name in ["BGE2SASRec","ColdBridge","ColdBridge+Blend","FULL_SYSTEM"]:
    if name not in all_results: continue
    r=all_results[name]; nd=float(r["NDCG@10"]); hr=float(r["HR@10"]); cv=float(r["CatalogCoverage"])
    d_nd=(nd-sas_nd)/sas_nd*100; d_hr=(hr-sas_hr)/sas_hr*100
    b_nd="✓" if nd>sas_nd else " "; b_hr="✓" if hr>sas_hr else " "
    print(f"★ {name:<23} {nd:>9.5f} {hr:>9.5f} {cv:>9.3f}   {b_nd}{d_nd:>+8.1f}%   {b_hr}{d_hr:>+8.1f}%")
print("="*70)
print("★ = our models   ✓ = beats Core SASRec on that metric")


In [ ]:
print("\n[SAVING] outputs …")
best_r=max(rows,key=lambda r:0.5*float(r["NDCG@10"])+0.5*float(r["HR@10"]))

res_df=pd.DataFrame(rows)
csv_p=RESULTS_DIR/"full_results.csv"
if csv_p.exists():
    old=pd.read_csv(csv_p)
    for col in res_df.columns:
        if col not in old.columns: old[col]=np.nan
    for col in old.columns:
        if col not in res_df.columns: res_df[col]=np.nan
    res_df=pd.concat([old,res_df],ignore_index=True)
res_df.to_csv(csv_p,index=False)

# pareto
mcols=["NDCG@10","HR@10","CatalogCoverage","ILD@10"]
vals=res_df[mcols].fillna(0).to_numpy(np.float64)
ip=np.ones(len(vals),bool)
for i in range(len(vals)):
    for j in range(len(vals)):
        if i!=j and np.all(vals[j]>=vals[i]) and np.any(vals[j]>vals[i]): ip[i]=False; break
res_df.loc[ip].to_csv(RESULTS_DIR/"pareto_front_v3.csv",index=False)
res_df.loc[ip].to_csv(RESULTS_DIR/"pareto_front_v2.csv",index=False)

# best_model.txt
pub_sas=PUBLISHED["SASRec"]; pub_llm=PUBLISHED["LLM2SASRec"]
bnd=float(best_r["NDCG@10"]); bhr=float(best_r["HR@10"])
lines=["=== BEST MODEL v3 ===",
       f"Model:      {best_r['model_name']}",
       f"Backbone:   PyTorch SASRec (2-layer attention, GELU, pre-norm)",
       f"Embedding:  BGE-M3 -> PCA {EMB_DIM}d","",
       f"NDCG@10:        {bnd:.6f}",
       f"  vs Core SASRec:      {(bnd-pub_sas['NDCG@10'])/pub_sas['NDCG@10']*100:+.2f}%  BEATS: {'YES' if bnd>pub_sas['NDCG@10'] else 'NO'}",
       f"  vs Core LLM2SASRec:  {(bnd-pub_llm['NDCG@10'])/pub_llm['NDCG@10']*100:+.2f}%  BEATS: {'YES' if bnd>pub_llm['NDCG@10'] else 'NO'}",
       f"HR@10:          {bhr:.6f}",
       f"  vs Core SASRec:      {(bhr-pub_sas['HR@10'])/pub_sas['HR@10']*100:+.2f}%  BEATS: {'YES' if bhr>pub_sas['HR@10'] else 'NO'}",
       f"  vs Core LLM2SASRec:  {(bhr-pub_llm['HR@10'])/pub_llm['HR@10']*100:+.2f}%  BEATS: {'YES' if bhr>pub_llm['HR@10'] else 'NO'}",
       f"CatalogCoverage:{float(best_r['CatalogCoverage']):.6f}",
       f"LongTail_HR@10: {float(best_r['LongTail_HR@10']):.6f}",
       f"ILD@10:         {float(best_r['ILD@10']):.6f}",
]
(RESULTS_DIR/"best_model.txt").write_text("\n".join(lines)+"\n")

# ablation md
adf=pd.DataFrame(abl).sort_values("step")
md=["# Ablation v3","","| Step | Model | NDCG@10 | HR@10 |","|---|---|---:|---:|"]
for _,r in adf.iterrows():
    md.append(f"| {int(r['step'])} | {r['model']} | {r['NDCG@10']:.6f} | {r['HR@10']:.6f} |")
(RESULTS_DIR/"ablation_v3.md").write_text("\n".join(md)+"\n")

# ablation + baseline comparison figure
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
ax.plot(adf["step"],adf["NDCG@10"],"o-",label="NDCG@10",color="steelblue")
ax.plot(adf["step"],adf["HR@10"],"s-",label="HR@10",color="darkorange")
for name,color,ls in [("SASRec","red","--"),("LLM2SASRec","purple","-."),("BERT4Rec","green",":")]:
    ax.axhline(PUBLISHED[name]["NDCG@10"],color=color,ls=ls,alpha=0.5,lw=1.2,label=f"{name} NDCG")
ax.set_xticks(adf["step"].tolist()); ax.set_xticklabels(adf["model"].tolist(),rotation=25,ha="right",fontsize=7)
ax.set_title("Ablation progression"); ax.legend(fontsize=7); ax.grid(alpha=0.25)

ax=axes[1]
bar_models=[r["model_name"] for r in rows if r["model_name"] in
            ["MostPopular","LLMSeqSim","SASRec","GRU4Rec","BERT4Rec","BGE2SASRec","ColdBridge","ColdBridge+Blend","FULL_SYSTEM"]]
bar_models=[m for m in ["MostPopular","LLMSeqSim","GRU4Rec","BERT4Rec","SASRec","BGE2SASRec","ColdBridge","ColdBridge+Blend","FULL_SYSTEM"] if m in {r["model_name"] for r in rows}]
brows={r["model_name"]:r for r in rows}
nd_vals=[float(brows[m]["NDCG@10"]) for m in bar_models]
hr_vals=[float(brows[m]["HR@10"]) for m in bar_models]
x=np.arange(len(bar_models)); w=0.35
bars1=ax.bar(x-w/2,nd_vals,w,label="NDCG@10",color="steelblue",alpha=0.8)
bars2=ax.bar(x+w/2,hr_vals,w,label="HR@10",color="darkorange",alpha=0.8)
ax.axhline(PUBLISHED["SASRec"]["NDCG@10"],color="red",ls="--",alpha=0.5,lw=1.2,label="Core SASRec NDCG")
ax.axhline(PUBLISHED["SASRec"]["HR@10"],  color="red",ls=":",alpha=0.5,lw=1.2,label="Core SASRec HR")
ax.set_xticks(x); ax.set_xticklabels(bar_models,rotation=30,ha="right",fontsize=7)
ax.set_title("All models comparison"); ax.legend(fontsize=7); ax.grid(alpha=0.25,axis="y")
plt.tight_layout(); plt.savefig(FIG_DIR/"all_models_comparison.png",dpi=150); plt.close()

print("Saved:", csv_p)
print("Saved:", RESULTS_DIR/"best_model.txt")
print("Saved:", FIG_DIR/"all_models_comparison.png")


In [ ]:
from IPython.display import display
csv_p=RESULTS_DIR/"full_results.csv"
if csv_p.exists():
    df=pd.read_csv(csv_p)
    display(df[["model_name","NDCG@10","HR@10","CatalogCoverage","LongTail_HR@10","ILD@10","training_time_sec"]]
              .sort_values("NDCG@10",ascending=False).head(20))

btxt=RESULTS_DIR/"best_model.txt"
if btxt.exists(): print(btxt.read_text())

amd=RESULTS_DIR/"ablation_v3.md"
if amd.exists(): print(amd.read_text())

pf=RESULTS_DIR/"pareto_front_v3.csv"
if pf.exists():
    print("\n=== Pareto front ===")
    display(pd.read_csv(pf)[["model_name","NDCG@10","HR@10","CatalogCoverage","ILD@10"]].head(15))
